# PrimePE — Phase 3: Real Language Modelling (H100 Edition)

> **Version:** v0.3.1-h100 | **Author:** Knack | **Date:** 2026-03-27  
> **Target:** Colab Pro H100 (80GB HBM3) — bf16 + FlashAttention-2 + torch.compile

## What this runs
| Step | Detail |
|------|--------|
| Model | 12-layer, d=512, 16 heads — ~125M params (GPT-2 medium scale) |
| Data | WikiText-103 (real language, not synthetic) |
| Training | 20K steps, bf16, FlashAttention-2, torch.compile |
| Eval seq lens | 512 / 2K / 4K / 8K / **16K** / **32K** |
| LitM context | 8K tokens — where RoPE aliasing actually bites |
| PE variants | sinusoidal, rope, prime_05, prime_10, zeta, hybrid, random_irr, learned, alibi |

## Why H100 unlocks the real test
- 80GB HBM3 → can hold a 125M model + 32K context batch comfortably in bf16
- FlashAttention-2 → O(N) memory for long contexts, no OOM at 32K
- torch.compile → ~40% throughput gain, brings total runtime to ~2-3hrs for full suite
- 32K context → **this is where sinusoidal PE aliases and ZetaPE shouldn't**

---
**Runtime estimate:** ~2.5–3.5 hrs for full 9-variant suite.  
Set `QUICK_RUN = True` in Cell 3 for a 3000-step smoke test (~25 min).


In [ ]:
# ─── Cell 1: Install ───────────────────────────────────────────────────────────
!pip install -q flash-attn --no-build-isolation
!pip install -q datasets transformers tokenizers matplotlib seaborn
print("Done.")

In [ ]:
# ─── Cell 2: GPU check ─────────────────────────────────────────────────────────
import torch, math, os, json, time, gc
import numpy as np

assert torch.cuda.is_available(), "No GPU — check runtime type"
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU:  {gpu}")
print(f"VRAM: {vram:.0f} GB")

IS_H100 = "H100" in gpu or "A100" in gpu
DTYPE   = torch.bfloat16 if IS_H100 else torch.float16
print(f"dtype: {DTYPE} | FlashAttn: {'yes' if IS_H100 else 'limited'}")

try:
    import flash_attn
    FLASH_AVAILABLE = True
    print(f"FlashAttention: {flash_attn.__version__} ✅")
except ImportError:
    FLASH_AVAILABLE = False
    print("FlashAttention: not available — using SDPA fallback")

DEVICE = torch.device("cuda")

In [ ]:
# ─── Cell 3: Configuration ─────────────────────────────────────────────────────
# Version: v0.3.1-h100
# Set QUICK_RUN = True for a fast smoke test (~25 min)

QUICK_RUN = False  # ← flip to True for smoke test

CFG = {
    # ── Model (GPT-2 medium scale) ────────────────────────────────────────────
    "d_model":      512,
    "n_heads":      16,
    "n_layers":     12,
    "ffn_dim":      2048,
    "dropout":      0.1,
    "vocab_size":   50257,

    # ── Training ──────────────────────────────────────────────────────────────
    "max_steps":    3000  if QUICK_RUN else 20000,
    "batch_size":   4     if QUICK_RUN else 8,
    "grad_accum":   4,
    "lr":           3e-4,
    "warmup_steps": 200   if QUICK_RUN else 1000,
    "weight_decay": 0.01,
    "clip_grad":    1.0,
    "eval_every":   500,
    "train_seq_len": 1024,   # train at 1K, eval at longer — tests extrapolation

    # ── Long-context eval ─────────────────────────────────────────────────────
    "eval_seq_lens":  [512, 1024, 2048, 4096, 8192, 16384, 32768]
                      if not QUICK_RUN else [512, 1024, 2048, 4096],

    # ── Lost-in-the-Middle benchmark ──────────────────────────────────────────
    "litm_context_len": 2048 if QUICK_RUN else 8192,
    "litm_positions":   [0.1, 0.25, 0.5, 0.75, 0.9],
    "litm_samples":     100  if QUICK_RUN else 500,

    # ── PE variants ───────────────────────────────────────────────────────────
    "pe_variants": [
        "sinusoidal",   # baseline
        "rope",         # baseline
        "prime_05",     # PrimePE α=0.5
        "prime_10",     # PrimePE α=1.0
        "zeta",         # ZetaPE
        "hybrid",       # Prime + Zeta
        "random_irr",   # CONTROL: random irrational
        "learned",      # CONTROL: learned from geometric init
        "alibi",        # ALiBi length-generalisation baseline
    ],

    # ── H100 flags ────────────────────────────────────────────────────────────
    "use_flash_attn":   FLASH_AVAILABLE,
    "use_compile":      True,
    "dtype":            DTYPE,
    "use_amp":          True,

    # ── Output ────────────────────────────────────────────────────────────────
    "results_dir": "/content/results",
    "seed":        42,
}

os.makedirs(CFG["results_dir"], exist_ok=True)
print(f"Mode: {'QUICK (smoke test)' if QUICK_RUN else 'FULL'}")
print(f"Model: {CFG['n_layers']}L × d{CFG['d_model']} × {CFG['n_heads']}h")
print(f"Steps: {CFG['max_steps']} | train_seq_len: {CFG['train_seq_len']}")
print(f"Max eval ctx: {max(CFG['eval_seq_lens'])} tokens")
print(f"LitM ctx: {CFG['litm_context_len']} tokens")
print(f"torch.compile: {CFG['use_compile']} | dtype: {CFG['dtype']}")

In [ ]:
# ─── Cell 4: PE frequency definitions ─────────────────────────────────────────
# Version: v0.3.1-h100

import torch.nn as nn
from typing import Optional

# ─── Zeta zeros — extended list (128 values for d=512) ────────────────────────
ZETA_ZEROS = [
    14.134725, 21.022040, 25.010858, 30.424876, 32.935061,
    37.586178, 40.918719, 43.327073, 48.005150, 49.773832,
    52.970321, 56.446247, 59.347044, 60.831778, 65.112544,
    67.079810, 69.546401, 72.067157, 75.704691, 77.144840,
    79.337375, 82.910381, 84.735493, 87.425275, 88.809111,
    92.491899, 94.651344, 95.870634, 98.831194, 101.317851,
    103.725538, 105.446623, 107.168611, 111.029535, 111.874659,
    114.320220, 116.226680, 118.790782, 121.370125, 122.946829,
    124.256818, 127.516683, 129.578704, 131.087688, 133.497737,
    134.756509, 138.116042, 139.736209, 141.123707, 143.111845,
    146.000982, 147.422765, 150.053521, 150.925257, 153.024693,
    156.112909, 157.597591, 158.849988, 161.188964, 163.030709,
    165.537069, 167.184439, 169.094515, 169.911976, 173.411536,
    174.754191, 176.441434, 178.377407, 179.916484, 182.207078,
    184.874467, 185.598783, 187.228922, 189.416158, 192.026656,
    193.079726, 195.265396, 196.876481, 198.015309, 201.264751,
    202.493594, 204.189671, 205.394697, 207.906258, 209.576509,
    211.690862, 213.347919, 214.547044, 216.169538, 219.067596,
    220.714918, 221.430705, 224.007000, 224.983324, 227.421444,
    229.337413, 231.250188, 231.987235, 233.693404, 236.524229,
    237.769820, 239.555477, 241.049157, 242.823271, 244.070898,
    247.136990, 248.101990, 249.573369, 251.014944, 253.069859,
    255.306388, 256.380713, 258.610439, 259.874406, 260.805801,
    263.573893, 265.557536, 266.614973, 267.921918, 269.969904,
    271.494350, 273.459609, 275.587492, 276.452049, 278.251218,
    279.229250, 282.465124, 283.211019, 284.835963, 285.752021,
]

def get_primes(n: int) -> list:
    primes, c = [], 2
    while len(primes) < n:
        if all(c % p != 0 for p in primes):
            primes.append(c)
        c += 1
    return primes

def get_zeta_freqs(n: int) -> torch.Tensor:
    """Return n normalised zeta-zero frequencies, extending with approximation if needed."""
    zz = list(ZETA_ZEROS[:n])
    if len(zz) < n:
        for i in range(len(zz), n):
            k = float(i + 1)
            zz.append((2 * math.pi * k) / (math.log(k) + 0.5))
    t = torch.tensor(zz[:n], dtype=torch.float32)
    return t / t[0]  # normalise to [1, ...]

def freq_spec(pe_name: str, d_model: int) -> dict:
    """Returns frequency tensor + flags for a PE variant."""
    half = d_model // 2
    if pe_name == "sinusoidal":
        return {"type": "additive", "learnable": False,
                "freqs": torch.exp(-torch.arange(0, half).float() * math.log(10000.0) / half)}
    elif pe_name == "rope":
        return {"type": "rope", "learnable": False,
                "freqs": 1.0 / (10000 ** (torch.arange(0, half, 2).float() / half))}
    elif pe_name == "alibi":
        return {"type": "alibi", "learnable": False, "freqs": None}
    elif pe_name == "prime_05":
        p = torch.tensor(get_primes(half), dtype=torch.float32)
        return {"type": "additive", "learnable": False, "freqs": 1.0 / (p ** 0.5)}
    elif pe_name == "prime_10":
        p = torch.tensor(get_primes(half), dtype=torch.float32)
        return {"type": "additive", "learnable": False, "freqs": 1.0 / p}
    elif pe_name == "zeta":
        return {"type": "additive", "learnable": False, "freqs": get_zeta_freqs(half)}
    elif pe_name == "hybrid":
        q = half // 2
        p = torch.tensor(get_primes(q), dtype=torch.float32)
        pf = 1.0 / (p ** 0.75)
        zf = get_zeta_freqs(half - q)
        return {"type": "additive", "learnable": False, "freqs": torch.cat([pf, zf])}
    elif pe_name == "random_irr":
        # Fractional parts of sqrt(prime) — provably irrational, zero number-theoretic structure
        p = torch.tensor(get_primes(half), dtype=torch.float32)
        freqs = p.sqrt().frac()
        freqs = freqs / freqs.max()
        return {"type": "additive", "learnable": False, "freqs": freqs}
    elif pe_name == "learned":
        freqs = 1.0 / (10000 ** (torch.arange(half).float() / half))
        return {"type": "additive", "learnable": True, "freqs": freqs}
    else:
        raise ValueError(f"Unknown PE: {pe_name}")

print("Frequency specs ready.")
for pe in CFG["pe_variants"]:
    spec = freq_spec(pe, CFG["d_model"])
    fmin = spec["freqs"].min().item() if spec["freqs"] is not None else None
    fmax = spec["freqs"].max().item() if spec["freqs"] is not None else None
    print(f"  {pe:<14} type={spec['type']:<10} learnable={spec['learnable']} "
          f"freq_range=[{fmin:.4f}, {fmax:.4f}]" if fmin is not None else f"  {pe}")

In [ ]:
# ─── Cell 5: Model — FlashAttention-2 + SDPA fallback ─────────────────────────
# Version: v0.3.1-h100

class AdditivePE(nn.Module):
    """Additive sinusoidal-style PE with arbitrary frequency set."""
    def __init__(self, d_model: int, freqs: torch.Tensor, max_len: int = 65536,
                 learnable: bool = False):
        super().__init__()
        self.d_model = d_model
        half = d_model // 2
        freqs = freqs[:half]
        if learnable:
            self.freqs = nn.Parameter(freqs.clone())
        else:
            self.register_buffer("freqs", freqs)
        self.learnable = learnable
        # Precompute for max_len if not learnable
        if not learnable:
            pos = torch.arange(max_len).float().unsqueeze(1)       # (L, 1)
            phases = pos * freqs.unsqueeze(0)                       # (L, half)
            pe = torch.cat([torch.sin(phases), torch.cos(phases)], dim=-1)  # (L, d)
            self.register_buffer("_pe_cache", pe)
        else:
            self._pe_cache = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        T = x.size(1)
        if self.learnable or self._pe_cache is None:
            pos = torch.arange(T, device=x.device).float().unsqueeze(1)
            phases = pos * self.freqs.unsqueeze(0)
            pe = torch.cat([torch.sin(phases), torch.cos(phases)], dim=-1)
        else:
            pe = self._pe_cache[:T]
        return x + pe.unsqueeze(0).to(x.dtype)


class Attention(nn.Module):
    """Multi-head attention: FlashAttn-2 if available, else PyTorch SDPA."""
    def __init__(self, d_model: int, n_heads: int, dropout: float,
                 attn_type: str = "standard",
                 rope_freqs: Optional[torch.Tensor] = None,
                 alibi_slopes: Optional[torch.Tensor] = None):
        super().__init__()
        self.n_heads  = n_heads
        self.head_dim = d_model // n_heads
        self.scale    = self.head_dim ** -0.5
        self.attn_type = attn_type
        self.qkv  = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.drop = nn.Dropout(dropout)
        if rope_freqs is not None:
            self.register_buffer("rope_freqs", rope_freqs)
        if alibi_slopes is not None:
            self.register_buffer("alibi_slopes", alibi_slopes)

    def _apply_rope(self, x: torch.Tensor) -> torch.Tensor:
        B, H, T, D = x.shape
        t = torch.arange(T, device=x.device).float()
        freqs = torch.outer(t, self.rope_freqs[:D//2])   # (T, D//2)
        cos = freqs.cos()[None, None].to(x.dtype)
        sin = freqs.sin()[None, None].to(x.dtype)
        x1, x2 = x[..., 0::2], x[..., 1::2]
        return torch.stack([x1*cos - x2*sin, x1*sin + x2*cos], dim=-1).flatten(-2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.head_dim)

        if FLASH_AVAILABLE and x.dtype in (torch.bfloat16, torch.float16):
            from flash_attn import flash_attn_qkvpacked_func
            if self.attn_type == "rope":
                q, k, v = qkv.unbind(2)                         # (B,T,H,D)
                q = self._apply_rope(q.permute(0,2,1,3)).permute(0,2,1,3)
                k = self._apply_rope(k.permute(0,2,1,3)).permute(0,2,1,3)
                qkv2 = torch.stack([q,k,v], dim=2)
                out = flash_attn_qkvpacked_func(qkv2, dropout_p=0.0, causal=True)
            else:
                out = flash_attn_qkvpacked_func(qkv, dropout_p=0.0, causal=True)
            out = out.reshape(B, T, C)
        else:
            # PyTorch SDPA fallback
            q, k, v = qkv.permute(2,0,3,1,4).unbind(0)         # each (B,H,T,D)
            if self.attn_type == "rope":
                q, k = self._apply_rope(q), self._apply_rope(k)
            if self.attn_type == "alibi":
                # Compute ALiBi bias
                pos = torch.arange(T, device=x.device)
                dist = (pos.unsqueeze(0) - pos.unsqueeze(1)).abs().float()  # (T,T)
                bias = -self.alibi_slopes.view(-1,1,1) * dist.unsqueeze(0)  # (H,T,T)
                # Causal mask
                causal = torch.triu(torch.full((T,T), float("-inf"), device=x.device), diagonal=1)
                attn_bias = bias + causal.unsqueeze(0)
                out = torch.nn.functional.scaled_dot_product_attention(
                    q, k, v, attn_mask=attn_bias, dropout_p=0.0)
            else:
                out = torch.nn.functional.scaled_dot_product_attention(
                    q, k, v, is_causal=True, dropout_p=0.0)
            out = out.transpose(1,2).reshape(B, T, C)

        return self.proj(self.drop(out))


class Block(nn.Module):
    def __init__(self, d_model: int, n_heads: int, ffn_dim: int, dropout: float,
                 attn_type: str, rope_freqs=None, alibi_slopes=None):
        super().__init__()
        self.attn  = Attention(d_model, n_heads, dropout, attn_type, rope_freqs, alibi_slopes)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn   = nn.Sequential(
            nn.Linear(d_model, ffn_dim), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model), nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


class PrimePEModel(nn.Module):
    """
    Decoder-only transformer with swappable PE.
    Version: v0.3.1-h100 | Author: Knack
    Change Log:
        v0.3.0  — Initial Phase 3 model
        v0.3.1  — H100: FlashAttn-2, bf16, SDPA fallback, pre-cached PE
    """
    def __init__(self, cfg: dict, pe_name: str):
        super().__init__()
        self.pe_name = pe_name
        d = cfg["d_model"]
        spec = freq_spec(pe_name, d)

        self.embed = nn.Embedding(cfg["vocab_size"], d)
        self.embed_drop = nn.Dropout(cfg["dropout"])

        # Additive PE (sinusoidal, prime_*, zeta, hybrid, random_irr, learned)
        self.pe = None
        if spec["type"] == "additive":
            self.pe = AdditivePE(d, spec["freqs"], learnable=spec["learnable"])

        # Build attention kwargs
        attn_type = spec["type"]  # "additive", "rope", "alibi"
        rope_freqs = spec["freqs"] if attn_type == "rope" else None
        if attn_type == "alibi":
            n = cfg["n_heads"]
            slopes = torch.tensor([2**(-8*i/n) for i in range(1, n+1)])
            alibi_slopes = slopes
        else:
            alibi_slopes = None

        self.blocks = nn.ModuleList([
            Block(d, cfg["n_heads"], cfg["ffn_dim"], cfg["dropout"],
                  attn_type if attn_type != "additive" else "standard",
                  rope_freqs, alibi_slopes)
            for _ in range(cfg["n_layers"])
        ])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, cfg["vocab_size"], bias=False)
        self.head.weight = self.embed.weight
        self._init()

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        x = self.embed(ids) * math.sqrt(ids.shape[-1]**0.5 + 1)  # scale embed
        if self.pe:
            x = self.pe(x)
        x = self.embed_drop(x)
        for block in self.blocks:
            x = block(x)
        return self.head(self.norm(x))

    def n_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Sanity check
torch.manual_seed(0)
_m = PrimePEModel(CFG, "zeta").to(DEVICE)
print(f"Model params: {_m.n_params()/1e6:.1f}M")
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=DTYPE):
    _o = _m(torch.randint(0, 1000, (2, 128)).to(DEVICE))
print(f"Forward pass OK — output shape: {_o.shape} ✅")
del _m, _o; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ─── Cell 6: Data loading ──────────────────────────────────────────────────────

from datasets import load_dataset
from transformers import GPT2TokenizerFast
from torch.utils.data import Dataset, DataLoader

print("Loading WikiText-103...")
raw = load_dataset("wikitext", "wikitext-103-raw-v1")
tok = GPT2TokenizerFast.from_pretrained("gpt2")
tok.pad_token = tok.eos_token

def _tokenise(split: str, seq_len: int) -> torch.Tensor:
    text = "\n".join(raw[split]["text"])
    ids  = torch.tensor(tok.encode(text), dtype=torch.long)
    n    = (len(ids) // seq_len) * seq_len
    return ids[:n].reshape(-1, seq_len)

class ChunkDataset(Dataset):
    def __init__(self, chunks): self.c = chunks
    def __len__(self): return len(self.c)
    def __getitem__(self, i): return self.c[i][:-1], self.c[i][1:]

SL = CFG["train_seq_len"] + 1
print(f"Tokenising (seq_len={SL-1})...")
train_ds = ChunkDataset(_tokenise("train",      SL))
val_ds   = ChunkDataset(_tokenise("validation", SL))

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True,
                          persistent_workers=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"], shuffle=False,
                          num_workers=4, pin_memory=True, drop_last=True,
                          persistent_workers=True)

print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,} sequences")

In [ ]:
# ─── Cell 7: Training loop ─────────────────────────────────────────────────────
# Version: v0.3.1-h100 — bf16 AMP, gradient accumulation, cosine LR

from itertools import cycle

def cosine_lr(step: int, cfg: dict) -> float:
    if step < cfg["warmup_steps"]:
        return cfg["lr"] * step / max(1, cfg["warmup_steps"])
    t = (step - cfg["warmup_steps"]) / max(1, cfg["max_steps"] - cfg["warmup_steps"])
    return cfg["lr"] * 0.5 * (1 + math.cos(math.pi * t))

@torch.no_grad()
def evaluate(model, loader, max_batches=100) -> float:
    model.eval()
    losses = []
    for i, (x, y) in enumerate(loader):
        if i >= max_batches: break
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            logits = model(x)
        loss = nn.functional.cross_entropy(logits.float().reshape(-1, CFG["vocab_size"]), y.reshape(-1))
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))

def train_variant(pe_name: str, cfg: dict) -> dict:
    print(f"\n{'═'*60}")
    print(f"  PE: {pe_name}")
    print(f"{'═'*60}")

    torch.manual_seed(cfg["seed"])
    model = PrimePEModel(cfg, pe_name).to(DEVICE)

    if cfg["use_compile"]:
        print("  Compiling model...")
        model = torch.compile(model, mode="reduce-overhead")

    print(f"  Params: {model.n_params() if hasattr(model, 'n_params') else '~125'}M")

    opt = torch.optim.AdamW(
        model.parameters(), lr=cfg["lr"],
        weight_decay=cfg["weight_decay"], betas=(0.9, 0.95), fused=True
    )
    scaler = torch.cuda.amp.GradScaler(enabled=(DTYPE == torch.float16))

    results = {
        "pe": pe_name, "steps": [], "train_loss": [], "val_loss": [],
        "final_val_loss": None, "final_val_ppl": None, "time_s": None,
    }

    it = cycle(train_loader)
    t0 = time.time()
    step = 0
    running_loss = 0.0
    opt.zero_grad()

    while step < cfg["max_steps"]:
        lr = cosine_lr(step, cfg)
        for g in opt.param_groups: g["lr"] = lr

        x, y = next(it)
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=DTYPE):
            logits = model(x)
            loss = nn.functional.cross_entropy(
                logits.float().reshape(-1, cfg["vocab_size"]),
                y.reshape(-1)
            ) / cfg["grad_accum"]

        scaler.scale(loss).backward()
        running_loss += loss.item()

        if (step + 1) % cfg["grad_accum"] == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["clip_grad"])
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)

            if step % cfg["eval_every"] == 0 or step == cfg["max_steps"] - 1:
                val = evaluate(model, val_loader)
                elapsed = time.time() - t0
                tokens_per_s = (step * cfg["batch_size"] * cfg["train_seq_len"]) / elapsed
                print(f"  step {step:6d} | train {running_loss:.4f} | val {val:.4f} "
                      f"| ppl {math.exp(val):.1f} | {tokens_per_s/1e3:.1f}K tok/s | {elapsed:.0f}s")
                results["steps"].append(step)
                results["train_loss"].append(running_loss)
                results["val_loss"].append(val)
                running_loss = 0.0

        step += 1

    final = evaluate(model, val_loader, max_batches=500)
    results["final_val_loss"] = final
    results["final_val_ppl"]  = math.exp(final)
    results["time_s"]         = time.time() - t0
    print(f"  ✅ Final — val_loss: {final:.4f}  PPL: {math.exp(final):.1f}  "
          f"time: {results['time_s']/60:.1f}min")

    # Save weights (unwrap compiled model)
    raw_model = model._orig_mod if hasattr(model, '_orig_mod') else model
    path = f"{cfg['results_dir']}/ckpt_{pe_name}.pt"
    torch.save(raw_model.state_dict(), path)
    results["ckpt"] = path

    del model; gc.collect(); torch.cuda.empty_cache()
    return results

print("Training loop ready.")

In [ ]:
# ─── Cell 8: Run all variants ──────────────────────────────────────────────────

all_results = {}
for pe in CFG["pe_variants"]:
    r = train_variant(pe, CFG)
    all_results[pe] = r
    with open(f"{CFG['results_dir']}/training_results.json", "w") as f:
        json.dump(all_results, f, indent=2, default=str)
    print(f"  Checkpoint saved after {pe}")

print("\n✅ All training complete.")

In [ ]:
# ─── Cell 9: Long-context perplexity ──────────────────────────────────────────
# Key test: does PPL degrade more slowly for ZetaPE as context grows?

@torch.no_grad()
def ppl_at_len(model, seq_len: int, n_batches: int = 50) -> float:
    chunks = _tokenise("test", seq_len + 1)
    ds     = ChunkDataset(chunks)
    loader = DataLoader(ds, batch_size=2, shuffle=False, drop_last=True)
    model.eval()
    losses = []
    for i, (x, y) in enumerate(loader):
        if i >= n_batches: break
        x, y = x.to(DEVICE), y.to(DEVICE)
        try:
            with torch.autocast(device_type="cuda", dtype=DTYPE):
                logits = model(x)
            loss = nn.functional.cross_entropy(
                logits.float().reshape(-1, CFG["vocab_size"]), y.reshape(-1))
            losses.append(loss.item())
        except RuntimeError as e:
            print(f"    OOM at len={seq_len}: skipping")
            break
    return math.exp(np.mean(losses)) if losses else float("nan")

print("Long-context PPL evaluation...")
ppl_results = {}

for pe in CFG["pe_variants"]:
    ckpt = all_results[pe].get("ckpt")
    if not ckpt: continue
    model = PrimePEModel(CFG, pe).to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))

    ppl_results[pe] = {}
    row = f"  {pe:<14}"
    for sl in CFG["eval_seq_lens"]:
        ppl = ppl_at_len(model, sl)
        ppl_results[pe][sl] = ppl
        row += f" | {sl//1024}K→{ppl:.1f}"
    print(row)

    del model; gc.collect(); torch.cuda.empty_cache()

with open(f"{CFG['results_dir']}/ppl_results.json", "w") as f:
    json.dump(ppl_results, f, indent=2)
print("\n✅ PPL eval complete.")

In [ ]:
# ─── Cell 10: Lost-in-the-Middle benchmark ────────────────────────────────────
# THE key number: retrieval accuracy at position 0.5 (exact middle)

@torch.no_grad()
def litm_benchmark(model, cfg: dict) -> dict:
    """
    Synthetic needle retrieval across context positions.
    Needle = a specific rare token placed at position frac * ctx_len.
    Query  = does the model predict the needle at that position?
    This directly measures the U-shaped attention degradation.
    """
    model.eval()
    ctx  = cfg["litm_context_len"]
    fracs = cfg["litm_positions"]
    results = {}

    for frac in fracs:
        needle_pos = max(1, int(frac * ctx) - 1)
        correct = 0

        for _ in range(cfg["litm_samples"]):
            # Random haystack tokens (common range)
            ids = torch.randint(200, 2000, (ctx,))
            # Inject needle — a token from a distinctive high range
            needle = torch.randint(40000, 45000, (1,)).item()
            ids[needle_pos] = needle

            inp = ids.unsqueeze(0).to(DEVICE)
            try:
                with torch.autocast(device_type="cuda", dtype=DTYPE):
                    logits = model(inp)   # (1, ctx, vocab)
                # Position needle_pos-1 should predict needle
                pred = logits[0, needle_pos - 1].argmax().item()
                if pred == needle:
                    correct += 1
            except RuntimeError:
                break

        acc = correct / cfg["litm_samples"]
        results[frac] = acc

    return results

print("Running Lost-in-the-Middle benchmark...")
print(f"Context: {CFG['litm_context_len']} tokens | Samples: {CFG['litm_samples']} per position")
litm_results = {}

for pe in CFG["pe_variants"]:
    ckpt = all_results[pe].get("ckpt")
    if not ckpt: continue
    model = PrimePEModel(CFG, pe).to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))

    litm = litm_benchmark(model, CFG)
    litm_results[pe] = litm

    mid = litm.get(0.5, 0)
    vals = " | ".join(f"{int(k*100)}%:{v:.1%}" for k, v in litm.items())
    flag = " ← KEY" if mid > 0 else ""
    print(f"  {pe:<14} {vals}{flag}")

    del model; gc.collect(); torch.cuda.empty_cache()

with open(f"{CFG['results_dir']}/litm_results.json", "w") as f:
    json.dump(litm_results, f, indent=2)
print("\n✅ LitM benchmark complete.")

In [ ]:
# ─── Cell 11: Plots ────────────────────────────────────────────────────────────

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

DARK, PANEL = "#0a0a0f", "#0d1117"
PE_COL = {
    "sinusoidal": "#90a4ae", "rope": "#546e7a",
    "prime_05":   "#b2ff59", "prime_10": "#76ff03",
    "zeta":       "#00e5ff", "hybrid": "#ff4081",
    "random_irr": "#ff9100", "learned": "#ffd740",
    "alibi":      "#ce93d8",
}

def dark_ax(figsize=(13,6)):
    fig, ax = plt.subplots(figsize=figsize, facecolor=DARK)
    ax.set_facecolor(PANEL)
    for spine in ["bottom","left"]: ax.spines[spine].set_color("#546e7a")
    for spine in ["top","right"]:   ax.spines[spine].set_visible(False)
    ax.tick_params(colors="#90a4ae")
    return fig, ax

def lbl(ax, txt, color="#90a4ae"):
    ax.set_xlabel(txt, color=color)
def ttl(ax, txt):
    ax.set_title(txt, color="#00e5ff", fontsize=13, pad=10)
def leg(ax):
    ax.legend(facecolor=PANEL, edgecolor="#546e7a", labelcolor="#e8eaf6", fontsize=8)
def savefig(fig, name):
    path = f"{CFG['results_dir']}/{name}"
    fig.tight_layout()
    fig.savefig(path, dpi=150, facecolor=DARK)
    plt.close()
    print(f"  Saved {name}")

# ── 1. Convergence ────────────────────────────────────────────────────────────
fig, ax = dark_ax()
for pe, r in all_results.items():
    if r.get("val_loss"):
        ax.plot(r["steps"], r["val_loss"], color=PE_COL.get(pe, "white"),
                label=pe, lw=2)
lbl(ax, "Step"); ax.set_ylabel("Val Loss", color="#90a4ae")
ttl(ax, "Convergence — All PE Variants"); leg(ax)
ax.grid(True, color="#1e293b", alpha=0.6)
savefig(fig, "convergence.png")

# ── 2. PPL vs context length ──────────────────────────────────────────────────
if ppl_results:
    fig, ax = dark_ax()
    for pe, lens in ppl_results.items():
        xs = sorted(lens.keys())
        ys = [lens[x] for x in xs]
        ax.plot(xs, ys, marker="o", color=PE_COL.get(pe, "white"),
                label=pe, lw=2, ms=6)
    ax.set_xscale("log"); ax.set_yscale("log")
    lbl(ax, "Context Length (tokens)"); ax.set_ylabel("Perplexity", color="#90a4ae")
    ttl(ax, "Perplexity vs Context Length — log/log")
    leg(ax); ax.grid(True, color="#1e293b", alpha=0.6)
    ax.axvline(10000, color="#ff4081", ls="--", alpha=0.4, lw=1, label="sinusoidal alias ~10K")
    savefig(fig, "ppl_vs_context.png")

# ── 3. LitM heatmap ───────────────────────────────────────────────────────────
if litm_results:
    fracs = CFG["litm_positions"]
    pes   = list(litm_results.keys())
    mat   = [[litm_results[p].get(f, 0) for f in fracs] for p in pes]
    fig, ax = plt.subplots(figsize=(10, max(4, len(pes)*0.7+1)), facecolor=DARK)
    ax.set_facecolor(PANEL)
    im = ax.imshow(mat, aspect="auto", cmap="plasma", vmin=0, vmax=1)
    ax.set_xticks(range(len(fracs)))
    ax.set_xticklabels([f"{int(f*100)}%" for f in fracs], color="#90a4ae")
    ax.set_yticks(range(len(pes)))
    ax.set_yticklabels(pes, color="#e8eaf6")
    ax.set_xlabel("Needle position (% of context)", color="#90a4ae")
    ax.set_title(f"Lost-in-the-Middle: Retrieval Accuracy @ {CFG['litm_context_len']} tokens",
                 color="#00e5ff", fontsize=12, pad=10)
    for i, p in enumerate(pes):
        for j, f in enumerate(fracs):
            v = litm_results[p].get(f, 0)
            ax.text(j, i, f"{v:.0%}", ha="center", va="center",
                    color="white", fontsize=9, fontweight="bold")
    fig.colorbar(im, ax=ax).ax.tick_params(colors="#90a4ae")
    savefig(fig, "litm_heatmap.png")

# ── 4. LitM curves ────────────────────────────────────────────────────────────
if litm_results:
    fig, ax = dark_ax()
    for pe, res in litm_results.items():
        xs = [f*100 for f in sorted(res.keys())]
        ys = [res[f] for f in sorted(res.keys())]
        ax.plot(xs, ys, marker="o", color=PE_COL.get(pe, "white"),
                label=pe, lw=2.5, ms=8)
    ax.axvline(50, color="#ff4081", ls="--", alpha=0.5, lw=1.5)
    ax.text(51, 0.05, "middle", color="#ff4081", fontsize=8)
    ax.set_ylim(0, 1.05)
    lbl(ax, "Needle position (% of context)")
    ax.set_ylabel("Retrieval accuracy", color="#90a4ae")
    ttl(ax, "Lost-in-the-Middle — Position Accuracy")
    leg(ax); ax.grid(True, color="#1e293b", alpha=0.6)
    savefig(fig, "litm_curves.png")

# ── 5. Final PPL bar chart ────────────────────────────────────────────────────
fig, ax = dark_ax((12, 5))
pes  = list(all_results.keys())
ppls = [all_results[p].get("final_val_ppl", float("nan")) for p in pes]
bars = ax.bar(pes, ppls, color=[PE_COL.get(p, "white") for p in pes], alpha=0.85)
for bar, ppl in zip(bars, ppls):
    if not math.isnan(ppl):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f"{ppl:.1f}", ha="center", va="bottom", color="#e8eaf6", fontsize=8)
ax.set_ylabel("Final Val PPL (lower = better)", color="#90a4ae")
ttl(ax, "Final Validation Perplexity — All PE Variants")
ax.grid(True, axis="y", color="#1e293b", alpha=0.6)
plt.xticks(rotation=30, ha="right", color="#e8eaf6")
savefig(fig, "final_ppl_bar.png")

print("\n✅ All plots saved.")

In [ ]:
# ─── Cell 12: Summary and analysis ────────────────────────────────────────────

print("\n" + "═"*80)
print("PHASE 3 RESULTS SUMMARY")
print("═"*80)
print(f"\nModel: {CFG['n_layers']}L × d{CFG['d_model']} × {CFG['n_heads']}h")
print(f"Data:  WikiText-103 | Steps: {CFG['max_steps']} | Train ctx: {CFG['train_seq_len']}")
print(f"LitM context: {CFG['litm_context_len']} tokens")

# Perplexity table
eval_lens = CFG["eval_seq_lens"]
header = f"{'PE':<14} {'ValPPL':<9}" + "".join(f" {'PPL@'+str(sl//1024)+'K':<9}" for sl in eval_lens)
print("\n" + header)
print("-" * len(header))
for pe in CFG["pe_variants"]:
    final = all_results.get(pe, {}).get("final_val_ppl", float("nan"))
    row = f"{pe:<14} {final:<9.1f}"
    for sl in eval_lens:
        ppl = ppl_results.get(pe, {}).get(sl, float("nan"))
        row += f" {ppl:<9.1f}"
    print(row)

# LitM table
fracs = CFG["litm_positions"]
print(f"\nLost-in-the-Middle @ {CFG['litm_context_len']} tokens")
hdr2 = f"{'PE':<14}" + "".join(f" {'@'+str(int(f*100))+'%':<9}" for f in fracs)
print(hdr2)
print("-" * len(hdr2))
for pe in CFG["pe_variants"]:
    res = litm_results.get(pe, {})
    row = f"{pe:<14}" + "".join(f" {res.get(f, 0):<9.1%}" for f in fracs)
    mid = res.get(0.5, 0)
    row += "  ← KEY" if mid > 0 else ""
    print(row)

# Key question
print("\n" + "─"*60)
print("KEY QUESTION: Does ZetaPE outperform random_irr at LitM@50%?")
zeta_mid = litm_results.get("zeta", {}).get(0.5, float("nan"))
rand_mid  = litm_results.get("random_irr", {}).get(0.5, float("nan"))
if not math.isnan(zeta_mid) and not math.isnan(rand_mid):
    delta = zeta_mid - rand_mid
    verdict = "YES — number-theoretic structure helps" if delta > 0.02 else \
              "MARGINAL" if delta > 0 else "NO — structure not the differentiator"
    print(f"  zeta@50%: {zeta_mid:.1%}  |  random_irr@50%: {rand_mid:.1%}  |  Δ={delta:+.1%}")
    print(f"  Verdict: {verdict}")

# Save everything
final_out = {
    "config": {k: str(v) for k, v in CFG.items()},
    "training":     all_results,
    "ppl_by_len":   ppl_results,
    "litm":         litm_results,
}
with open(f"{CFG['results_dir']}/phase3_FINAL.json", "w") as f:
    json.dump(final_out, f, indent=2, default=str)
print(f"\nAll results → {CFG['results_dir']}/phase3_FINAL.json")

In [ ]:
# ─── Cell 13: Download ─────────────────────────────────────────────────────────

from google.colab import files
import glob

to_dl = (glob.glob(f"{CFG['results_dir']}/*.json") +
         glob.glob(f"{CFG['results_dir']}/*.png"))

print(f"Downloading {len(to_dl)} files...")
for f in sorted(to_dl):
    print(f"  {os.path.basename(f)}")
    files.download(f)

print("\n✅ Done.")